In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import torch
import numpy as np
from pathlib import Path

from fmri_mapping.embedding.evaluation import accuracy_cosine_similarity, compute_rsa
from fmri_mapping.embedding.ops import split_repetitions
from fmri_mapping.io.nsd import get_resource
from tqdm.notebook import tqdm, trange

In [23]:
df_stimuli = get_resource("stimulus")

min_repetitions = 3
stimuli_list = df_stimuli.query("repetition == @min_repetitions -1 and shared and exists").groupby("nsd_id").size().reset_index().rename(columns={0: "n_subjects"}).query("n_subjects == 8").nsd_id.tolist()
len(stimuli_list)

515

In [54]:

results = []
for depth in trange(1, 3 + 1):
    folder = Path(f"../scripts/mlp_embeddings_dimensions_depth-{depth}")
    # All that end in standarized.pt
    files = list(folder.glob("*sub-01.pt"))
    file_templates = [file.name.replace("sub-01", r"sub-{subject:02d}") for file in files]
  
    for filename in tqdm(file_templates, leave=False):
        for subject in range(1, 8 + 1):

            data = torch.load(folder / filename.format(subject=subject))
            Z = data["Z"]

            df_reps = split_repetitions(subject=subject, shuffle_indexes=False, min_exists=3)
            df_reps = df_reps.query("shared and nsd_id in @stimuli_list")


            Z1 = Z[df_reps.subject_index_1.values]#.cuda()
            Z2 = Z[df_reps.subject_index_2.values]#.cuda()
            Z3 = Z[df_reps.subject_index_3.values]#.cuda()


            views = [(1, Z1), (2, Z2), (3, Z3)]
            for i, Zi in views:
                for j, Zj in views:
                    if i < j:
                        acc, cosine, mean_rank = accuracy_cosine_similarity(Zi, Zj)
                        rsa = compute_rsa(Zi, Zj)
                        results.append({
                            "subject": subject,
                            "view_i": i,
                            "view_j": j,
                            "accuracy": acc,
                            "cosine_similarity": cosine,
                            "mean_rank": mean_rank,
                            "rsa": rsa,
                            "template": filename,
                            "folder": folder.name,
                        })


df_results = pd.DataFrame(results)
df_results

  0%|          | 0/3 [00:00<?, ?it/s]

  0%|          | 0/221 [00:00<?, ?it/s]

  0%|          | 0/213 [00:00<?, ?it/s]

  0%|          | 0/65 [00:00<?, ?it/s]

,subject,view_i,view_j,accuracy,cosine_similarity,mean_rank,rsa,template,folder
0,1,1,2,0.994175,0.679480,1.021359,0.612016,ws_mlp_pca-768_depth-1_280_1024_sub-{subject:0...,mlp_embeddings_dimensions_depth-1
1,1,1,3,0.992233,0.672313,1.021359,0.605763,ws_mlp_pca-768_depth-1_280_1024_sub-{subject:0...,mlp_embeddings_dimensions_depth-1
2,1,2,3,0.990291,0.672810,1.064078,0.602608,ws_mlp_pca-768_depth-1_280_1024_sub-{subject:0...,mlp_embeddings_dimensions_depth-1
3,2,1,2,0.986408,0.692546,1.126214,0.632681,ws_mlp_pca-768_depth-1_280_1024_sub-{subject:0...,mlp_embeddings_dimensions_depth-1
4,2,1,3,0.984466,0.681387,1.033010,0.621127,ws_mlp_pca-768_depth-1_280_1024_sub-{subject:0...,mlp_embeddings_dimensions_depth-1
...,...,...,...,...,...,...,...,...,...
11971,7,1,3,0.574757,0.387105,12.093204,0.552194,ws_mlp_pca-1024_depth-3_280_128_sub-{subject:0...,mlp_embeddings_dimensions_depth-3
11972,7,2,3,0.559223,0.387668,13.733980,0.547410,ws_mlp_pca-1024_depth-3_280_128_sub-{subject:0...,mlp_embeddings_dimensions_depth-3
11973,8,1,2,0.467961,0.332862,21.062136,0.484541,ws_mlp_pca-1024_depth-3_280_128_sub-{subject:0...,mlp_embeddings_dimensions_depth-3
11974,8,1,3,0.431068,0.317295,18.984467,0.468357,ws_mlp_pca-1024_depth-3_280_128_sub-{subject:0...,mlp_embeddings_dimensions_depth-3


In [58]:
df_results

,subject,view_i,view_j,accuracy,cosine_similarity,mean_rank,rsa,template,folder
0,1,1,2,0.994175,0.679480,1.021359,0.612016,ws_mlp_pca-768_depth-1_280_1024_sub-{subject:0...,mlp_embeddings_dimensions_depth-1
1,1,1,3,0.992233,0.672313,1.021359,0.605763,ws_mlp_pca-768_depth-1_280_1024_sub-{subject:0...,mlp_embeddings_dimensions_depth-1
2,1,2,3,0.990291,0.672810,1.064078,0.602608,ws_mlp_pca-768_depth-1_280_1024_sub-{subject:0...,mlp_embeddings_dimensions_depth-1
3,2,1,2,0.986408,0.692546,1.126214,0.632681,ws_mlp_pca-768_depth-1_280_1024_sub-{subject:0...,mlp_embeddings_dimensions_depth-1
4,2,1,3,0.984466,0.681387,1.033010,0.621127,ws_mlp_pca-768_depth-1_280_1024_sub-{subject:0...,mlp_embeddings_dimensions_depth-1
...,...,...,...,...,...,...,...,...,...
11971,7,1,3,0.574757,0.387105,12.093204,0.552194,ws_mlp_pca-1024_depth-3_280_128_sub-{subject:0...,mlp_embeddings_dimensions_depth-3
11972,7,2,3,0.559223,0.387668,13.733980,0.547410,ws_mlp_pca-1024_depth-3_280_128_sub-{subject:0...,mlp_embeddings_dimensions_depth-3
11973,8,1,2,0.467961,0.332862,21.062136,0.484541,ws_mlp_pca-1024_depth-3_280_128_sub-{subject:0...,mlp_embeddings_dimensions_depth-3
11974,8,1,3,0.431068,0.317295,18.984467,0.468357,ws_mlp_pca-1024_depth-3_280_128_sub-{subject:0...,mlp_embeddings_dimensions_depth-3


In [55]:
df_results.to_parquet("dimensions-ablations-within-encoder.parquet", index=False)

In [60]:
df_results_g = df_results.groupby(["template", "folder"]).aggregate({"accuracy": "mean", "cosine_similarity": "mean", "mean_rank": "mean", "rsa": "mean"}).reset_index()
df_results_g["pca_dim"] = df_results_g.template.str.split("_").str[2].str.split("-").str[1].astype(int)
df_results_g["depth"] = df_results_g.folder.str.split("_").str[3].str.split("-").str[1].astype(int)

df_results_g

,template,folder,accuracy,cosine_similarity,mean_rank,rsa,pca_dim,depth
0,ws_mlp_pca-1024_depth-1_256_1024_sub-{subject:...,mlp_embeddings_dimensions_depth-1,0.817152,0.569134,5.263511,0.537749,1024,1
1,ws_mlp_pca-1024_depth-1_256_128_sub-{subject:0...,mlp_embeddings_dimensions_depth-1,0.778964,0.405901,7.032120,0.434815,1024,1
2,ws_mlp_pca-1024_depth-1_256_256_sub-{subject:0...,mlp_embeddings_dimensions_depth-1,0.803236,0.473428,5.831877,0.501811,1024,1
3,ws_mlp_pca-1024_depth-1_256_512_sub-{subject:0...,mlp_embeddings_dimensions_depth-1,0.811165,0.532936,5.377670,0.538835,1024,1
4,ws_mlp_pca-1024_depth-1_256_768_sub-{subject:0...,mlp_embeddings_dimensions_depth-1,0.816505,0.557347,5.278641,0.541099,1024,1
...,...,...,...,...,...,...,...,...
494,ws_mlp_pca-768_depth_88_768_sub-{subject:02d}.pt,mlp_embeddings_dimensions_depth-1,0.818366,0.614627,5.218204,0.522259,768,1
495,ws_mlp_pca-768_depth_96_1024_sub-{subject:02d}.pt,mlp_embeddings_dimensions_depth-1,0.818932,0.615237,5.206958,0.521790,768,1
496,ws_mlp_pca-768_depth_96_256_sub-{subject:02d}.pt,mlp_embeddings_dimensions_depth-1,0.826214,0.583202,5.222249,0.516840,768,1
497,ws_mlp_pca-768_depth_96_512_sub-{subject:02d}.pt,mlp_embeddings_dimensions_depth-1,0.823139,0.603996,5.183900,0.524320,768,1


In [61]:
df_results_g.sort_values("mean_rank")

,template,folder,accuracy,cosine_similarity,mean_rank,rsa,pca_dim,depth
458,ws_mlp_pca-768_depth_120_768_sub-{subject:02d}.pt,mlp_embeddings_dimensions_depth-1,0.822087,0.602398,5.177589,0.526813,768,1
462,ws_mlp_pca-768_depth_128_768_sub-{subject:02d}.pt,mlp_embeddings_dimensions_depth-1,0.822977,0.599636,5.179045,0.528098,768,1
461,ws_mlp_pca-768_depth_128_512_sub-{subject:02d}.pt,mlp_embeddings_dimensions_depth-1,0.824272,0.588479,5.181553,0.529276,768,1
497,ws_mlp_pca-768_depth_96_512_sub-{subject:02d}.pt,mlp_embeddings_dimensions_depth-1,0.823139,0.603996,5.183900,0.524320,768,1
454,ws_mlp_pca-768_depth_112_768_sub-{subject:02d}.pt,mlp_embeddings_dimensions_depth-1,0.822330,0.605289,5.185356,0.525680,768,1
...,...,...,...,...,...,...,...,...
129,ws_mlp_pca-1024_depth-3_360_128_sub-{subject:0...,mlp_embeddings_dimensions_depth-3,0.654773,0.409120,9.536327,0.592108,1024,3
438,ws_mlp_pca-768_depth-3_360_128_sub-{subject:02...,mlp_embeddings_dimensions_depth-3,0.645307,0.415424,9.663107,0.594634,768,3
433,ws_mlp_pca-768_depth-3_320_128_sub-{subject:02...,mlp_embeddings_dimensions_depth-3,0.654935,0.414385,9.669175,0.591399,768,3
134,ws_mlp_pca-1024_depth-3_384_128_sub-{subject:0...,mlp_embeddings_dimensions_depth-3,0.642314,0.395829,10.213916,0.578886,1024,3


In [72]:
folder = Path(f"../scripts/ablations_rois")
template = "ws_mlp_{roi}_128_768_sub-{subject:02d}.pt"
rois = [ "V1-V4", "V1", "surface", ]
results = []
# scripts/ablations_rois/ws_mlp_V1_128_768_sub-01.pt
for roi in tqdm(rois):
    for subject in range(1, 8 + 1):
        data = torch.load(folder / template.format(roi=roi, subject=subject))
        Z = data["Z"]

        df_reps = split_repetitions(subject=subject, shuffle_indexes=False, min_exists=3)
        df_reps = df_reps.query("shared and nsd_id in @stimuli_list")


        Z1 = Z[df_reps.subject_index_1.values]#.cuda()
        Z2 = Z[df_reps.subject_index_2.values]#.cuda()
        Z3 = Z[df_reps.subject_index_3.values]#.cuda()


        views = [(1, Z1), (2, Z2), (3, Z3)]
        for i, Zi in views:
            for j, Zj in views:
                if i < j:
                    acc, cosine, mean_rank = accuracy_cosine_similarity(Zi, Zj)
                    rsa = compute_rsa(Zi, Zj)
                    results.append({
                        "subject": subject,
                        "view_i": i,
                        "view_j": j,
                        "accuracy": acc,
                        "cosine_similarity": cosine,
                        "mean_rank": mean_rank,
                        "rsa": rsa,
                        "template": template,
                        "folder": folder.name,
                        "roi": roi,
                    })

df_results_rois = pd.DataFrame(results)
df_results_rois

  0%|          | 0/3 [00:00<?, ?it/s]

'../scripts/ablations_rois/ws_linear_V1-V4_768_128_sub-01.pt'

In [74]:
df_results_rois = pd.DataFrame(results)
df_results_rois.to_parquet("ablations-rois-within-encoder.parquet", index=False)
df_results_rois.groupby("roi").aggregate({"accuracy": "mean", "cosine_similarity": "mean", "mean_rank": "mean", "rsa": "mean"}).reset_index().sort_values("mean_rank")

,roi,accuracy,cosine_similarity,mean_rank,rsa
2,surface,0.805987,0.587785,5.587540,0.517474
1,V1-V4,0.742395,0.555038,8.200081,0.474387
0,V1,0.541505,0.445548,23.416748,0.372942
